# 06 — Сборка `mart_users`

**Источник:** `data/processed/mart_events.parquet`

Гранулярность: **одна строка = один `user_id`**.

В этом ноутбуке `train.csv`, `questions.csv` и `lectures.csv` повторно не используются: вся агрегация строится непосредственно из уже готовой `mart_events`.

### Зафиксированные правила

- минимальная история для `first_20_accuracy`, `last_20_accuracy` и `progress_20`: **40 вопросов**;
- `progressing`: `progress_20 >= 0.05`;
- `declining`: `progress_20 <= -0.05`;
- остальные пользователи с достаточной историей: `stable`;
- пользователи с менее чем 40 вопросами получают `progress_segment = NaN`;
- `lecture_per_question = lecture_views / questions_count`;
- `explanation_rate` рассчитывается из `prior_question_had_explanation` и трактуется как доля вопросных событий, для которых этот исторический флаг равен `True`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Пути проекта
PROJECT_DIR = Path("..")  
EVENTS_PATH = PROJECT_DIR / "data" / "processed" / "mart_events.parquet"
USERS_PATH = PROJECT_DIR / "data" / "processed" / "mart_users.parquet"

MIN_QUESTIONS_FOR_PROGRESS = 40
PROGRESS_THRESHOLD = 0.05

print("mart_events:", EVENTS_PATH)
print("mart_users:", USERS_PATH)


mart_events: ../data/processed/mart_events.parquet
mart_users: ../data/processed/mart_users.parquet


## 1. Загрузка `mart_events`

In [3]:
events = pd.read_parquet(EVENTS_PATH)

print("Размер mart_events:", events.shape)
display(events.head())
display(events.dtypes)


Размер mart_events: (101230332, 34)


,row_id,user_id,timestamp,content_id,content_type_id,content_kind,task_container_id,user_answer,answered_correctly,prior_question_elapsed_time,...,rolling_accuracy_5,rolling_accuracy_20,time_since_last_event,lectures_before,questions_before,correct_answers_before,user_accuracy_before,error_streak,correct_streak,session_id
0,8418,40828,0,7900,0,question,0,0.0,1.0,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0,0,1.0
1,8419,40828,26382,7876,0,question,1,2.0,0.0,17000.0,...,1.000000,1.000000,26382.0,0.0,1.0,1.0,1.000000,0,1,1.0
2,8420,40828,51949,175,0,question,2,2.0,1.0,20000.0,...,0.500000,0.500000,25567.0,0.0,2.0,1.0,0.500000,1,0,1.0
3,8421,40828,75274,1278,0,question,3,3.0,1.0,21000.0,...,0.666667,0.666667,23325.0,0.0,3.0,2.0,0.666667,0,1,1.0
4,8422,40828,174812,2065,0,question,4,2.0,1.0,17000.0,...,0.750000,0.750000,99538.0,0.0,4.0,3.0,0.750000,0,2,1.0


row_id                              int64
user_id                             int32
timestamp                           int64
content_id                          int32
content_type_id                      int8
content_kind                          str
task_container_id                   int32
user_answer                       float64
answered_correctly                float64
prior_question_elapsed_time       float64
prior_question_had_explanation     object
question_id                       float64
bundle_id                         float64
correct_answer                    float64
question_part                     float64
tags                                  str
lecture_id                        float64
tag                               float64
lecture_part                      float64
type_of                               str
part                                int16
event_number                        int64
question_number                   float64
previous_correct                  

## 2. Проверка структуры

Перед агрегацией проверяем, что `mart_events` содержит признаки, необходимые для `mart_users`.
Если какого-то столбца нет, ноутбук остановится здесь, а не создаст неполную витрину.


In [4]:
required_columns = [
    "user_id",
    "timestamp",
    "content_type_id",
    "answered_correctly",
    "question_id",
    "part",
    "tags",
    "session_id",
    "time_since_last_event",
    "correct_streak",
    "error_streak",
    "prior_question_elapsed_time",
    "prior_question_had_explanation",
]

missing_columns = sorted(set(required_columns) - set(events.columns))

if missing_columns:
    raise ValueError(
        "В mart_events отсутствуют необходимые столбцы: "
        + ", ".join(missing_columns)
    )

print("Все необходимые столбцы присутствуют.")


Все необходимые столбцы присутствуют.


## 3. Разделение question / lecture events

`content_type_id = 0` — вопрос, `content_type_id = 1` — лекция.

Метрики успешности считаются **только по вопросам**.


In [5]:
is_question = events["content_type_id"].eq(0)
is_lecture = events["content_type_id"].eq(1)
events.loc[is_lecture, "answered_correctly"].value_counts(dropna=False)

answered_correctly
NaN    1959032
Name: count, dtype: int64

In [6]:
events.loc[
    is_lecture,
    ["content_type_id", "answered_correctly"]
].head(20)

,content_type_id,answered_correctly
90,1,NaN
203,1,NaN
251,1,NaN
265,1,NaN
280,1,NaN
291,1,NaN
309,1,NaN
436,1,NaN
510,1,NaN
589,1,NaN


In [7]:
is_question = events["content_type_id"].eq(0)
is_lecture = events["content_type_id"].eq(1)

questions = events.loc[is_question].copy()

print("Question events:", is_question.sum())
print("Lecture events:", is_lecture.sum())

assert questions["answered_correctly"].isin([0, 1]).all()
assert events.loc[is_lecture, "answered_correctly"].isna().all()


Question events: 99271300
Lecture events: 1959032


## 4. Активность пользователей

In [8]:
users = (
    events.groupby("user_id")
    .agg(
        events_count=("user_id", "size"),
        sessions_count=("session_id", "nunique"),
        unique_parts=("part", "nunique"),
    )
    .reset_index()
)

question_activity = (
    questions.groupby("user_id")
    .agg(
        questions_count=("answered_correctly", "size"),
        correct_answers=("answered_correctly", "sum"),
        unique_questions=("question_id", "nunique"),
    )
    .reset_index()
)

lecture_activity = (
    events.loc[is_lecture]
    .groupby("user_id")
    .agg(
        lectures_count=("lecture_id", "size") if "lecture_id" in events.columns
        else ("content_id", "size"),
        lecture_views=("user_id", "size"),
    )
    .reset_index()
)

users = users.merge(
    question_activity,
    on="user_id",
    how="left",
    validate="one_to_one",
)

users = users.merge(
    lecture_activity,
    on="user_id",
    how="left",
    validate="one_to_one",
)

# Пользователи, у которых не было вопросов/лекций.
for col in [
    "questions_count",
    "correct_answers",
    "unique_questions",
    "lectures_count",
    "lecture_views",
]:
    users[col] = users[col].fillna(0)

for col in [
    "questions_count",
    "correct_answers",
    "unique_questions",
    "lectures_count",
    "lecture_views",
]:
    users[col] = users[col].astype("int64")

users["incorrect_answers"] = (
    users["questions_count"] - users["correct_answers"]
)

users["accuracy"] = (
    users["correct_answers"]
    / users["questions_count"].replace(0, np.nan)
)

users["lecture_per_question"] = (
    users["lecture_views"]
    / users["questions_count"].replace(0, np.nan)
)

display(users.head())


,user_id,events_count,sessions_count,unique_parts,questions_count,correct_answers,unique_questions,lectures_count,lecture_views,incorrect_answers,accuracy,lecture_per_question
0,115,46,2,5,46,32,46,0,0,14,0.695652,0.000000
1,124,30,1,7,30,7,30,0,0,23,0.233333,0.000000
2,2746,20,1,2,19,11,17,1,1,8,0.578947,0.052632
3,5382,128,19,3,125,84,120,3,3,41,0.672000,0.024000
4,8623,112,14,4,109,70,104,3,3,39,0.642202,0.027523


## 5. Количество уникальных тем

В `tags` у вопроса может быть несколько тегов, записанных через пробел.
Считаем число различных тегов, встреченных пользователем.


In [9]:
tag_counts = (
    questions[["user_id", "tags"]]
    .dropna()
    .assign(tags=lambda x: x["tags"].astype(str).str.split())
    .explode("tags")
    .groupby("user_id")["tags"]
    .nunique()
    .rename("unique_tags")
    .reset_index()
)

users = users.merge(
    tag_counts,
    on="user_id",
    how="left",
    validate="one_to_one",
)

users["unique_tags"] = users["unique_tags"].fillna(0).astype("int64")


## 6. First / last accuracy

Рассчитываем accuracy первых и последних 10/20 вопросов каждого пользователя.

Для прогресса используются только пользователи минимум с 40 вопросами.


In [10]:
questions = questions.sort_values(
    ["user_id", "timestamp"],
    kind="mergesort"
)

first10_accuracy = (
    questions.groupby("user_id")
    .head(10)
    .groupby("user_id")["answered_correctly"]
    .mean()
    .rename("first_10_accuracy")
)

first20_accuracy = (
    questions.groupby("user_id")
    .head(20)
    .groupby("user_id")["answered_correctly"]
    .mean()
    .rename("first_20_accuracy")
)

last10_accuracy = (
    questions.groupby("user_id")
    .tail(10)
    .groupby("user_id")["answered_correctly"]
    .mean()
    .rename("last_10_accuracy")
)

last20_accuracy = (
    questions.groupby("user_id")
    .tail(20)
    .groupby("user_id")["answered_correctly"]
    .mean()
    .rename("last_20_accuracy")
)

accuracy_features = pd.concat(
    [
        first10_accuracy,
        first20_accuracy,
        last10_accuracy,
        last20_accuracy,
    ],
    axis=1,
).reset_index()

users = users.merge(
    accuracy_features,
    on="user_id",
    how="left",
    validate="one_to_one",
)


## 7. Streaks

В `mart_events` уже есть `correct_streak` и `error_streak`.
Поэтому повторно считать последовательности не нужно.

Берём максимальную длину серии для каждого пользователя.


In [11]:
streak_features = (
    questions.groupby("user_id")
    .agg(
        best_correct_streak=("correct_streak", "max"),
        max_error_streak=("error_streak", "max"),
    )
    .reset_index()
)

users = users.merge(
    streak_features,
    on="user_id",
    how="left",
    validate="one_to_one",
)

users["best_correct_streak"] = (
    users["best_correct_streak"].fillna(0).astype("int64")
)

users["max_error_streak"] = (
    users["max_error_streak"].fillna(0).astype("int64")
)


## 8. Прогресс пользователя

`progress_20 = last_20_accuracy - first_20_accuracy`.

Пользователи с менее чем 40 вопросами не классифицируются по прогрессу.


In [12]:
enough_history = users["questions_count"].ge(
    MIN_QUESTIONS_FOR_PROGRESS
)

users["progress_20"] = np.where(
    enough_history,
    users["last_20_accuracy"] - users["first_20_accuracy"],
    np.nan,
)

users["progress_segment"] = np.select(
    [
        users["progress_20"].ge(PROGRESS_THRESHOLD),
        users["progress_20"].le(-PROGRESS_THRESHOLD),
    ],
    [
        "progressing",
        "declining",
    ],
    default="stable",
)

users.loc[
    ~enough_history,
    "progress_segment"
] = pd.NA

print(users["progress_segment"].value_counts(dropna=False))


progress_segment
NaN            184388
progressing    121334
declining       52425
stable          35509
Name: count, dtype: int64


## 9. Временные метрики

`learning_duration` — время между первым и последним событием пользователя.

`median_time_between_events` — медианный промежуток между последовательными событиями.

`avg_questions_per_session` — среднее число решённых вопросов за одну сессию.


In [13]:
time_features = (
    events.groupby("user_id")
    .agg(
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        median_time_between_events=("time_since_last_event", "median"),
    )
    .reset_index()
)

time_features["learning_duration"] = (
    time_features["last_timestamp"]
    - time_features["first_timestamp"]
)

time_features["learning_duration_days"] = (
    time_features["learning_duration"]
    / (24 * 60 * 60 * 1000)
)

questions_per_session = (
    questions.groupby(["user_id", "session_id"])
    .size()
    .groupby("user_id")
    .mean()
    .rename("avg_questions_per_session")
    .reset_index()
)

time_features = time_features.merge(
    questions_per_session,
    on="user_id",
    how="left",
    validate="one_to_one",
)

users = users.merge(
    time_features,
    on="user_id",
    how="left",
    validate="one_to_one",
)

users.drop(
    columns=["first_timestamp", "last_timestamp"],
    inplace=True,
)


## 10. Время решения вопросов

In [14]:
elapsed_features = (
    questions.groupby("user_id")["prior_question_elapsed_time"]
    .agg(
        median_question_elapsed_time="median",
        mean_question_elapsed_time="mean",
    )
    .reset_index()
)

users = users.merge(
    elapsed_features,
    on="user_id",
    how="left",
    validate="one_to_one",
)


## 11. Использование объяснений

В Riiid `prior_question_had_explanation` — это исторический признак:
он относится к предыдущему набору вопросов.

Поэтому `explanation_rate` здесь означает долю вопросных событий, где этот
флаг был `True`, а не буквально «доля вопросов, после которых пользователь
открыл объяснение».


In [15]:
explanation_rate = (
    questions.groupby("user_id")["prior_question_had_explanation"]
    .apply(lambda s: s.fillna(False).mean())
    .rename("explanation_rate")
    .reset_index()
)

users = users.merge(
    explanation_rate,
    on="user_id",
    how="left",
    validate="one_to_one",
)


## 12. Приведение порядка столбцов

Сначала идут идентификатор и активность, затем успешность, прогресс,
временные характеристики и использование материалов.


In [16]:
final_columns = [
    "user_id",

    # Активность
    "questions_count",
    "lectures_count",
    "events_count",
    "sessions_count",
    "unique_questions",
    "unique_parts",
    "unique_tags",

    # Успешность
    "correct_answers",
    "incorrect_answers",
    "accuracy",
    "first_10_accuracy",
    "first_20_accuracy",
    "last_10_accuracy",
    "last_20_accuracy",
    "best_correct_streak",
    "max_error_streak",

    # Прогресс
    "progress_20",
    "progress_segment",

    # Время
    "median_question_elapsed_time",
    "mean_question_elapsed_time",
    "median_time_between_events",
    "learning_duration",
    "learning_duration_days",
    "avg_questions_per_session",

    # Образовательные материалы
    "lecture_views",
    "lecture_per_question",
    "explanation_rate",
]

missing_final = sorted(set(final_columns) - set(users.columns))

if missing_final:
    raise ValueError(
        "Не удалось сформировать столбцы: "
        + ", ".join(missing_final)
    )

users = users[final_columns]

display(users.head())
print("Размер mart_users:", users.shape)


,user_id,questions_count,lectures_count,events_count,sessions_count,unique_questions,unique_parts,unique_tags,correct_answers,incorrect_answers,...,progress_segment,median_question_elapsed_time,mean_question_elapsed_time,median_time_between_events,learning_duration,learning_duration_days,avg_questions_per_session,lecture_views,lecture_per_question,explanation_rate
0,115,46,0,46,2,46,5,34,32,14,...,declining,20000.0,19933.311111,23476.0,668090043,7.732524,23.000000,0,0.000000,0.130435
1,124,30,0,30,1,30,7,35,7,23,...,NaN,21000.0,18793.000000,10438.0,571323,0.006613,30.000000,0,0.000000,0.000000
2,2746,19,1,20,1,17,2,28,11,8,...,NaN,17500.0,18055.555556,28831.0,835457,0.009670,19.000000,1,0.052632,0.578947
3,5382,125,3,128,19,120,3,82,84,41,...,stable,25000.0,36048.387097,144372.0,2101551456,24.323512,6.578947,3,0.024000,0.904000
4,8623,109,3,112,14,104,4,83,70,39,...,progressing,20000.0,26107.407407,42467.0,862338736,9.980772,7.785714,3,0.027523,0.880734


Размер mart_users: (393656, 28)


## 13. Финальные проверки качества

In [17]:
# Одна строка = один пользователь
assert users["user_id"].is_unique

# Accuracy в [0, 1]
assert users["accuracy"].dropna().between(0, 1).all()
assert users["first_10_accuracy"].dropna().between(0, 1).all()
assert users["first_20_accuracy"].dropna().between(0, 1).all()
assert users["last_10_accuracy"].dropna().between(0, 1).all()
assert users["last_20_accuracy"].dropna().between(0, 1).all()

# Корректность числа ответов
assert (
    users["correct_answers"] + users["incorrect_answers"]
    == users["questions_count"]
).all()

# Общее число событий не меньше числа вопросов
assert (
    users["events_count"] >= users["questions_count"]
).all()

# Прогресс только при достаточной истории
assert users.loc[
    users["questions_count"] < MIN_QUESTIONS_FOR_PROGRESS,
    "progress_20"
].isna().all()

assert users.loc[
    users["questions_count"] < MIN_QUESTIONS_FOR_PROGRESS,
    "progress_segment"
].isna().all()

# Progress не может выходить за [-1, 1]
assert users["progress_20"].dropna().between(-1, 1).all()

# Доли
assert users["lecture_per_question"].dropna().ge(0).all()
assert users["explanation_rate"].dropna().between(0, 1).all()

# Бесконечные значения
numeric_cols = users.select_dtypes(include=np.number).columns
assert not np.isinf(
    users[numeric_cols].to_numpy()
).any()

print("Все проверки mart_users пройдены.")


Все проверки mart_users пройдены.


## 14. Сохранение витрины

Формат Parquet используется для дальнейшего подключения к дашборду и аналитическим ноутбукам.


In [18]:
USERS_PATH.parent.mkdir(parents=True, exist_ok=True)

users.to_parquet(
    USERS_PATH,
    index=False
)

print(f"mart_users сохранена: {USERS_PATH}")
print(f"Строк: {len(users):,}")
print(f"Столбцов: {len(users.columns)}")


mart_users сохранена: ../data/processed/mart_users.parquet
Строк: 393,656
Столбцов: 28


## 15. Контрольный просмотр

После сохранения читаем файл обратно и проверяем, что он действительно открывается.


In [19]:
mart_users_check = pd.read_parquet(USERS_PATH)

assert mart_users_check.shape == users.shape
assert mart_users_check["user_id"].is_unique

print("Файл успешно читается обратно.")
display(mart_users_check.head())
display(mart_users_check.describe(include="all").T)


Файл успешно читается обратно.


,user_id,questions_count,lectures_count,events_count,sessions_count,unique_questions,unique_parts,unique_tags,correct_answers,incorrect_answers,...,progress_segment,median_question_elapsed_time,mean_question_elapsed_time,median_time_between_events,learning_duration,learning_duration_days,avg_questions_per_session,lecture_views,lecture_per_question,explanation_rate
0,115,46,0,46,2,46,5,34,32,14,...,declining,20000.0,19933.311111,23476.0,668090043,7.732524,23.000000,0,0.000000,0.130435
1,124,30,0,30,1,30,7,35,7,23,...,NaN,21000.0,18793.000000,10438.0,571323,0.006613,30.000000,0,0.000000,0.000000
2,2746,19,1,20,1,17,2,28,11,8,...,NaN,17500.0,18055.555556,28831.0,835457,0.009670,19.000000,1,0.052632,0.578947
3,5382,125,3,128,19,120,3,82,84,41,...,stable,25000.0,36048.387097,144372.0,2101551456,24.323512,6.578947,3,0.024000,0.904000
4,8623,109,3,112,14,104,4,83,70,39,...,progressing,20000.0,26107.407407,42467.0,862338736,9.980772,7.785714,3,0.027523,0.880734


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,393656.0,NaN,NaN,NaN,1076358303.90729,620131874.576781,115.0,538759611.75,1077717363.5,1613533216.25,2147482888.0
questions_count,393656.0,NaN,NaN,NaN,252.17779,734.721108,1.0,30.0,40.0,154.0,17609.0
lectures_count,393656.0,NaN,NaN,NaN,4.976507,15.964786,0.0,0.0,0.0,2.0,397.0
events_count,393656.0,NaN,NaN,NaN,257.154297,747.550934,1.0,30.0,41.0,157.0,17917.0
sessions_count,393656.0,NaN,NaN,NaN,15.10308,44.544327,1.0,1.0,3.0,10.0,2374.0
unique_questions,393656.0,NaN,NaN,NaN,220.667362,589.913962,1.0,29.0,40.0,142.0,11218.0
unique_parts,393656.0,NaN,NaN,NaN,4.63932,2.309854,1.0,2.0,5.0,7.0,7.0
unique_tags,393656.0,NaN,NaN,NaN,61.730285,43.577858,1.0,34.0,43.0,85.0,188.0
correct_answers,393656.0,NaN,NaN,NaN,165.740207,519.376811,0.0,11.0,23.0,95.0,14300.0
incorrect_answers,393656.0,NaN,NaN,NaN,86.437583,232.326614,0.0,12.0,22.0,57.0,8112.0
